
<div dir="rtl" align="right">

# سؤال ۱: تعیین واحد تحلیل (Unit of Analysis) و ساختار داده

## سؤال

با توجه به هدف پروژه Online Retail، واحد تحلیل مناسب برای آماده‌سازی داده چیست؟

مشخص کنید هر ردیف از دیتاست خام نشان‌دهنده‌ی چه چیزی است و برای مراحل بعدی پروژه به چه سطحی از داده نیاز داریم.

موارد زیر را بررسی کنید:

1. آیا هر ردیف نشان‌دهنده‌ی یک مشتری، یک فاکتور یا یک قلم کالای ثبت‌شده در فاکتور است؟
2. تفاوت سطح Transaction-Level و Customer-Level چیست؟
3. چرا برای محاسبه‌ی ویژگی‌های LRFM و انجام Customer Segmentation باید داده‌ها در سطح مشتری تجمیع شوند؟
4. چرا برای Classification مبتنی بر رفتار زمانی مشتری، باید امکان ساخت چند نمونه از یک مشتری در ReferenceDateهای متفاوت را داشته باشیم؟
5. ساختار داده‌ی موردنیاز برای هر یک از دو مسیر Clustering و Classification چیست؟

## چرایی و دلیل

دیتاست خام Online Retail در سطح ردیف‌های تراکنش محصول ثبت شده است؛ بنابراین هر ردیف الزاماً نماینده‌ی یک مشتری مستقل نیست.

یک مشتری می‌تواند در چندین فاکتور و هر فاکتور در چندین ردیف محصول حضور داشته باشد.

اگر بدون توجه به سطح تحلیل، مستقیماً روی ردیف‌های خام مدل‌سازی انجام دهیم، ممکن است یک مشتری چندین بار به‌عنوان مشاهدات مستقل در نظر گرفته شود و رفتار واقعی مشتری به‌درستی نمایش داده نشود.

## هدف

تعیین سطح صحیح تحلیل و طراحی ساختار داده‌ی متناسب با دو مسئله‌ی اصلی پروژه:

- Customer Segmentation با الگوریتم K-Means
- Churn Classification با الگوریتم XGBoost

## منطق تصمیم‌گیری

برای Clustering، هر مشتری در یک ReferenceDate مشخص باید یک مشاهده داشته باشد و ویژگی‌های LRFM و رفتاری او از تراکنش‌های معتبر استخراج شوند.

برای Classification، هر مشاهده نماینده‌ی رفتار یک مشتری تا یک ReferenceDate مشخص است و برچسب Churn با استفاده از رفتار آینده‌ی همان مشتری در Prediction Horizon ساخته می‌شود.

بنابراین ممکن است یک مشتری در تاریخ‌های مرجع متفاوت، چند نمونه‌ی تاریخی داشته باشد؛ اما این نمونه‌ها نباید به‌صورت تصادفی بین Train و Test تقسیم شوند.

## فایده

- جلوگیری از تحلیل اشتباه در سطح ردیف تراکنش
- فراهم‌کردن بستر صحیح محاسبه‌ی LRFM
- جلوگیری از تکرار غیرضروری مشتریان در Clustering
- امکان ساخت نمونه‌های زمانی برای Classification
- کاهش خطر Data Leakage
- ایجاد ساختار داده‌ی صحیح برای دو مسیر مستقل مدل‌سازی

</div>


---

In [94]:
import pandas as pd


df = pd.read_csv("../Online Retail.csv")

df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/10 8:26,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/10 8:26,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/10 8:28,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/10 8:28,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/10 8:34,1.69,13047.0,United Kingdom


In [95]:
# Check dataset dimensions
print("Dataset Shape:", df.shape)

# Check column names
print("\nColumns:")
print(df.columns.tolist())

# Check data types
print("\nData Types:")
print(df.dtypes)

Dataset Shape: (541909, 8)

Columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data Types:
InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object


In [96]:
# Count unique customers and invoices
print(
    "Unique Customers:",
    df["CustomerID"].nunique()
)

print(
    "Unique Invoices:",
    df["InvoiceNo"].nunique()
)

print(
    "Total Transaction Lines:",
    len(df)
)

Unique Customers: 4372
Unique Invoices: 25900
Total Transaction Lines: 541909



<div dir="rtl" align="right">

### نتایج اجرا و تحلیل سؤال اول: بررسی ساختار اولیه دیتاست

**تحلیل نتایج:**

دیتاست در سطح سطر تراکنش قرار دارد؛ یعنی هر ردیف نشان‌دهنده یک سطر ثبت‌شده از کالا در یک فاکتور است و الزاماً نماینده یک مشتری یا یک فاکتور کامل نیست.

همچنین نوع داده `CustomerID` به‌صورت `float64` ثبت شده است؛ درحالی‌که این ستون از نظر مفهومی یک شناسه است و نباید به‌عنوان متغیر عددی پیوسته تحلیل شود. وجود مقادیر گمشده می‌تواند یکی از دلایل انتخاب این نوع داده باشد.

ستون `InvoiceDate` نیز از نوع رشته است و برای انجام محاسبات زمانی باید در مراحل بعدی به نوع تاریخ و زمان تبدیل شود.

**تصمیم نهایی:**

در این مرحله هیچ ردیفی حذف یا تجمیع نمی‌شود و نوع داده‌ها نیز هنوز تغییر نمی‌کند.

ساختار داده برای ادامه پروژه به سه سطح تقسیم می‌شود:

1. داده خام در سطح سطر تراکنش.
2. داده مشتری‌محور برای الگوریتم K-Means و بخش‌بندی مشتریان.
3. داده مشتری در تاریخ مرجع (`Customer × ReferenceDate`) برای ساخت برچسب Churn و آموزش XGBoost.

**نتیجه:**

ساختار اولیه دیتاست شناسایی شد و مشخص گردید که برای اجرای دو مسئله Clustering و Classification به دو ساختار داده متفاوت در مراحل بعدی نیاز داریم.

</div>


---


<div dir="rtl" align="right">

### سؤال دوم: مقادیر گمشده و رکوردهای تکراری چه وضعیتی دارند؟

**چرایی و دلیل:**

وجود مقادیر گمشده و رکوردهای تکراری می‌تواند باعث ایجاد خطا در محاسبات و تحلیل‌های بعدی شود. برای مثال، نبودن `CustomerID` مانع انتساب یک تراکنش به مشتری مشخص می‌شود و رکوردهای تکراری ممکن است باعث شوند مقدار خرید یا تعداد تراکنش‌ها بیشتر از مقدار واقعی محاسبه شود.

بااین‌حال، هر مقدار گمشده یا هر سطر مشابهی الزاماً به معنی داده نامعتبر نیست. بنابراین قبل از تصمیم‌گیری درباره حذف، جایگزینی یا نگهداری داده‌ها، باید میزان و الگوی این مشکلات بررسی شود.

**هدف:**

شناسایی تعداد و درصد مقادیر گمشده در هر ستون، بررسی میزان تکرار کامل ردیف‌ها و مشخص‌کردن ستون‌هایی که بیشترین مشکل کیفیت داده را دارند.

**منطق تصمیم‌گیری:**

- مقادیر گمشده را به تفکیک ستون بررسی می‌کنیم.
- فقط درصد گمشدگی را ملاک تصمیم‌گیری قرار نمی‌دهیم؛ اهمیت هر ستون در تحلیل نیز مهم است.
- رکوردهای تکراری کامل را شناسایی می‌کنیم، اما هنوز حذفشان نمی‌کنیم.
- وضعیت `CustomerID` را جداگانه بررسی می‌کنیم، زیرا این ستون برای تحلیل رفتار مشتری و ساخت هر دو دیتاست ضروری است.

**فایده:**

نتایج این بررسی مبنای تصمیم‌گیری درباره نحوه برخورد با داده‌های ناقص و تکراری در مراحل بعدی Data Preparation خواهد بود.

</div>


In [97]:

import pandas as pd

# 1. تعداد و درصد مقادیر گمشده در هر ستون
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
})

missing_summary = missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
)

print("Missing Values Summary:")
display(missing_summary)


# 2. بررسی تعداد و درصد رکوردهای تکراری کامل
duplicate_count = df.duplicated().sum()
duplicate_percentage = (duplicate_count / len(df)) * 100

print("\nDuplicate Records:")
print(f"Duplicate rows: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")


# 3. بررسی وضعیت CustomerID
missing_customer_ids = df["CustomerID"].isna().sum()
known_customer_ids = df["CustomerID"].nunique()

print("\nCustomerID Analysis:")
print(f"Missing CustomerID rows: {missing_customer_ids:,}")
print(f"Known unique customers: {known_customer_ids:,}")
print(
    f"Missing CustomerID percentage: "
    f"{df['CustomerID'].isna().mean() * 100:.2f}%"
)


Missing Values Summary:


,Missing_Count,Missing_Percentage
CustomerID,135080,24.93
Description,1454,0.27
StockCode,0,0.00
InvoiceNo,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
Country,0,0.00



Duplicate Records:
Duplicate rows: 5,268
Duplicate percentage: 0.97%

CustomerID Analysis:
Missing CustomerID rows: 135,080
Known unique customers: 4,372
Missing CustomerID percentage: 24.93%



<div dir="rtl" align="right">

**تحلیل نتایج:**

بیشترین میزان مقادیر گمشده مربوط به `CustomerID` است. تعداد 135,080 ردیف فاقد شناسه مشتری هستند که معادل 24.93 درصد کل ردیف‌های دیتاست است.

این مسئله برای پروژه اهمیت زیادی دارد؛ زیرا هر دو مسیر اصلی مدل‌سازی در سطح مشتری انجام می‌شوند:

- در K-Means برای محاسبه ویژگی‌های رفتاری و LRFM باید تراکنش‌ها به مشتری مشخصی منتسب باشند.
- در XGBoost برای ساخت نمونه‌های آموزشی و تعیین وضعیت Churn باید سابقه خرید هر مشتری قابل شناسایی باشد.

بنابراین ردیف‌های فاقد `CustomerID` نمی‌توانند وارد تحلیل مشتری‌محور شوند؛ اما برای تحلیل عمومی تراکنش‌ها همچنان قابل استفاده‌اند.

ستون `Description` نیز دارای 1,454 مقدار گمشده، معادل 0.27 درصد است. از آنجا که `StockCode` در تمام ردیف‌ها موجود است، نبود توضیحات کالا به‌تنهایی دلیل کافی برای حذف تراکنش نیست.

**۲. نتایج بررسی رکوردهای تکراری**

- تعداد ردیف‌های کاملاً تکراری اضافه: 5,268
- درصد ردیف‌های تکراری اضافه: 0.97 درصد
- تعداد ردیف‌های دارای CustomerID گمشده: 135,080
- تعداد مشتریان یکتای شناسایی‌شده: 4,372

**تحلیل نتایج:**

بررسی اولیه نشان داد که 5,268 ردیف کاملاً تکراری در دیتاست وجود دارد. این ردیف‌ها ممکن است باعث محاسبه چندباره مقدار خرید و در نتیجه انحراف ویژگی‌های مشتری شوند.

همچنین باید توجه داشت که تعداد 135,080 مربوط به ردیف‌های فاقد شناسه مشتری است، نه تعداد مشتریان گمشده یا ناشناس.

**۳. تصمیم‌گیری درباره مقادیر گمشده**

- `CustomerID`: عدم جایگزینی با شناسه ساختگی؛ نگهداری در داده خام و کنارگذاری از تحلیل مشتری‌محور.
- `Description`: نگهداری ردیف‌های دارای مقدار گمشده؛ عدم حذف تراکنش صرفاً به دلیل نبود توضیحات کالا.
- سایر ستون‌ها: در این بررسی مقدار گمشده مشاهده نشد.

**۴. تصمیم‌گیری درباره رکوردهای تکراری**

رکوردهای کاملاً تکراری باید از نسخه تحلیلی حذف شوند، اما دیتاست خام بدون تغییر حفظ می‌شود.

این حذف فقط برای ردیف‌هایی انجام می‌شود که تمام ستون‌های آن‌ها با یکدیگر یکسان باشند.

**نتیجه:**

دو مشکل اصلی کیفیت داده شناسایی شد:

1. نبود شناسه مشتری در 24.93 درصد از ردیف‌های تراکنش.
2. وجود 5,268 ردیف تکراری اضافه.

تصمیم اولیه برای مدیریت این مشکلات مشخص شد؛ اما اجرای حذف رکوردهای تکراری و ساخت نسخه پاک‌سازی‌شده در مرحله اجرایی بعدی انجام می‌شود.

</div>


---


<div dir="rtl" align="right">

### سؤال سوم: آیا رکوردهای تکراری باید حذف شوند؟

**چرایی و دلیل:**

رکوردهای تکراری کامل ممکن است باعث محاسبه چندباره یک تراکنش شوند. این مسئله می‌تواند روی مجموع مقدار خرید، ارزش خرید مشتری، تعداد اقلام و در نهایت ویژگی‌های مورد استفاده در مدل‌های Clustering و Classification اثر بگذارد.

بااین‌حال، پیش از حذف باید ماهیت رکوردهای تکراری را بررسی کنیم. ممکن است یک کالا در یک فاکتور چند بار با اطلاعات یکسان ثبت شده باشد؛ بنابراین صرفاً مشاهده مقادیر یکسان، بدون بررسی زمینه داده، برای نتیجه‌گیری قطعی کافی نیست.

**هدف:**

شناسایی ردیف‌های تکراری، مشاهده نمونه‌های آن‌ها، بررسی محل وقوع تکرار و اندازه‌گیری اثر احتمالی حذف آن‌ها بر مقدار خام تراکنش‌ها.

**منطق تصمیم‌گیری:**

- ردیف‌های تکراری را با مقایسه تمام ستون‌ها شناسایی می‌کنیم.
- نمونه ردیف‌های تکراری را مشاهده می‌کنیم.
- تعداد فاکتورهای درگیر در تکرار را بررسی می‌کنیم.
- اثر ردیف‌های تکراری اضافه را بر مجموع Quantity و حاصل‌ضرب Quantity در UnitPrice محاسبه می‌کنیم.
- پس از بررسی، درباره حذف ردیف‌های کاملاً تکراری تصمیم می‌گیریم.

**فایده:**

جلوگیری از محاسبه چندباره تراکنش‌ها و کاهش احتمال انحراف ویژگی‌های مشتری در مراحل بعدی Data Preparation.

</div>


In [98]:

# 1. شناسایی تمام ردیف‌هایی که در گروه‌های تکراری قرار دارند
duplicate_groups = df[df.duplicated(keep=False)].copy()

# 2. شناسایی فقط نسخه‌های اضافه رکوردهای تکراری
# این همان 5268 ردیفی است که در بررسی قبلی شمارش کردیم.
extra_duplicates = df[df.duplicated(keep="first")].copy()

print("Duplicate Analysis:")
print(f"Rows involved in duplicate groups: {len(duplicate_groups):,}")
print(f"Extra duplicate rows: {len(extra_duplicates):,}")
print(f"Number of affected invoices: {extra_duplicates['InvoiceNo'].nunique():,}")


# 3. مشاهده نمونه‌ای از رکوردهای تکراری
print("\nDuplicate Samples:")
display(
    duplicate_groups.sort_values(
        by=["InvoiceNo", "StockCode"]
    ).head(20)
)


# 4. محاسبه اثر رکوردهای تکراری اضافه بر داده خام
extra_duplicates["RawLineValue"] = (
    extra_duplicates["Quantity"] * extra_duplicates["UnitPrice"]
)

print("\nPotential Impact of Extra Duplicate Rows:")
print(f"Extra duplicate Quantity: {extra_duplicates['Quantity'].sum():,}")
print(
    "Extra duplicate raw line value: "
    f"{extra_duplicates['RawLineValue'].sum():,.2f}"
)
print(
    "Absolute raw line value: "
    f"{extra_duplicates['RawLineValue'].abs().sum():,.2f}"
)


# 5. بررسی فاکتورهایی که بیشترین تعداد ردیف تکراری اضافه را دارند
print("\nInvoices with the Most Extra Duplicate Rows:")

display(
    extra_duplicates.groupby("InvoiceNo")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .rename("Extra_Duplicate_Rows")
    .to_frame()
)


Duplicate Analysis:
Rows involved in duplicate groups: 10,147
Extra duplicate rows: 5,268
Number of affected invoices: 1,933

Duplicate Samples:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,12/1/10 11:45,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,12/1/10 11:45,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,12/1/10 11:45,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,12/1/10 11:45,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,12/1/10 11:45,2.10,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,12/1/10 11:45,2.10,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,12/1/10 11:45,2.95,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,12/1/10 11:45,2.95,17908.0,United Kingdom
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,12/1/10 11:49,1.65,17920.0,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,12/1/10 11:49,1.65,17920.0,United Kingdom



Potential Impact of Extra Duplicate Rows:
Extra duplicate Quantity: 13,948
Extra duplicate raw line value: 21,740.98
Absolute raw line value: 27,406.50

Invoices with the Most Extra Duplicate Rows:


,Extra_Duplicate_Rows
InvoiceNo,
555524,34
547651,31
579458,28
574481,27
572103,27
577482,26
537781,25
552285,24
575117,21



<div dir="rtl" align="right">

### نتایج اجرا و تحلیل: بررسی ماهیت و اثر رکوردهای تکراری

در بررسی دقیق‌تر رکوردهای تکراری، دو شاخص متفاوت محاسبه شد:

- تعداد ردیف‌های عضو گروه‌های تکراری: 10,147
- تعداد نسخه‌های تکراری اضافه: 5,268

تفاوت این دو مقدار به دلیل وجود گروه‌هایی است که یک رکورد در آن‌ها بیش از دو بار تکرار شده است. در محاسبه نسخه‌های اضافه، اولین نمونه از هر گروه حفظ می‌شود.

**۱. گستردگی تکرارها**

تعداد فاکتورهای درگیر در تکرار: 1,933

بنابراین رکوردهای تکراری فقط به چند فاکتور محدود نیستند و در تعداد قابل‌توجهی از فاکتورها مشاهده شده‌اند.

**۳. تصمیم نهایی**

نسخه‌های کاملاً تکراری از داده تحلیلی حذف می‌شوند و اولین نمونه از هر گروه باقی می‌ماند.

ردیف‌هایی که فقط در بعضی ویژگی‌ها مانند `InvoiceNo` یا `StockCode` مشترک هستند، تکراری کامل محسوب نمی‌شوند و نباید با این روش حذف شوند.

**نتیجه:**

با توجه به وجود 5,268 نسخه تکراری اضافه و اثر بالقوه آن‌ها بر محاسبات، حذف تکرارهای کامل از نسخه تحلیلی تصمیم مناسبی است.

</div>


In [99]:

# ساخت نسخه جدید بدون رکوردهای کاملاً تکراری
df_dedup = df.drop_duplicates(keep="first").copy()

# بررسی نتیجه
print("Original Dataset Shape:", df.shape)
print("Dataset Shape After Deduplication:", df_dedup.shape)

print(
    "Removed Rows:",
    len(df) - len(df_dedup)
)

print(
    "Remaining Duplicate Rows:",
    df_dedup.duplicated().sum()
)


Original Dataset Shape: (541909, 8)
Dataset Shape After Deduplication: (536641, 8)
Removed Rows: 5268
Remaining Duplicate Rows: 0



<div dir="rtl" align="right">

### نتایج اجرا و تحلیل سؤال سوم: حذف رکوردهای کاملاً تکراری

پس از شناسایی و بررسی رکوردهای تکراری، تصمیم گرفته شد نسخه‌های کاملاً تکراری از داده تحلیلی حذف شوند و اولین نمونه از هر گروه حفظ شود.

برای اجرای این تصمیم، از متد `drop_duplicates()` با مقدار `keep="first"` استفاده شد. همچنین برای جلوگیری از تغییر دیتاست خام، یک کپی مستقل از داده‌ها ایجاد شد.

**تحلیل نتایج:**

تعداد ردیف‌های دیتاست از 541,909 به 536,641 کاهش یافت. این کاهش دقیقاً برابر با 5,268 ردیف تکراری اضافه شناسایی‌شده در مرحله بررسی بود.

پس از اجرای عملیات، مقدار `duplicated().sum()` برابر صفر شد. بنابراین هیچ ردیف کاملاً تکراری بر اساس تمام ستون‌ها در نسخه جدید باقی نمانده است.

تعداد ستون‌ها بدون تغییر باقی ماند؛ زیرا عملیات حذف تکرار فقط روی ردیف‌های دیتاست انجام شده است.

**تصمیم نهایی:**

نسخه تحلیلی بدون رکوردهای کاملاً تکراری با نام `df_dedup` ساخته شد. دیتاست خام با نام `df` نیز برای حفظ قابلیت بازبینی و مقایسه نگهداری شد.

**فایده برای پروژه:**

این عملیات از محاسبه چندباره ردیف‌های یکسان جلوگیری می‌کند و احتمال انحراف ویژگی‌هایی مانند مجموع خرید، تعداد کالاهای خریداری‌شده و ارزش تراکنش مشتری را کاهش می‌دهد.

این نسخه، مبنای ادامه مراحل پاک‌سازی داده خواهد بود.

</div>


---


<div dir="rtl" align="right">

### سؤال چهارم: فاکتورهای لغوشده و تراکنش‌های دارای مقادیر منفی یا صفر چه وضعیتی دارند؟

**چرایی و دلیل:**

در دیتاست Online Retail تمام ردیف‌ها الزاماً نشان‌دهنده خرید موفق مشتری نیستند. برخی فاکتورها لغو شده‌اند، برخی ردیف‌ها دارای تعداد کالای منفی هستند و در بعضی موارد قیمت واحد نیز منفی یا صفر است.

این مقادیر می‌توانند نشان‌دهنده رویدادهای متفاوتی مانند لغو سفارش، مرجوعی کالا، اصلاح اطلاعات یا ثبت‌های مالی خاص باشند. بنابراین حذف تمام ردیف‌های دارای مقادیر منفی بدون بررسی، ممکن است باعث از بین رفتن اطلاعات ارزشمند شود.

این موضوع برای پروژه اهمیت ویژه‌ای دارد؛ زیرا ویژگی‌های رفتاری مشتری مانند Monetary، Frequency، Recency و در نهایت برچسب Churn باید بر اساس تعریف مشخصی از خرید معتبر ساخته شوند.

**هدف:**

شناسایی و اندازه‌گیری موارد زیر:

1. فاکتورهای لغوشده بر اساس پیشوند `C` در `InvoiceNo`.
2. ردیف‌های دارای `Quantity` منفی یا صفر.
3. ردیف‌های دارای `UnitPrice` منفی یا صفر.
4. میزان هم‌پوشانی این شرایط با یکدیگر.
5. نمونه‌هایی از هر گروه برای بررسی ماهیت داده‌ها.

**منطق تصمیم‌گیری:**

- فاکتورهایی که شماره آن‌ها با `C` شروع می‌شود، به‌عنوان فاکتور لغوشده شناسایی می‌شوند.
- تعداد منفی می‌تواند نشان‌دهنده مرجوعی یا ثبت اصلاحی باشد؛ اما به‌تنهایی برای اثبات مرجوعی کافی نیست.
- قیمت منفی ممکن است مربوط به تعدیلات یا ثبت‌های خاص باشد و باید جداگانه بررسی شود.
- مقدار صفر در تعداد یا قیمت نیز باید از مقادیر منفی تفکیک شود.
- هم‌پوشانی شرایط بررسی می‌شود تا از شمارش چندباره یک ردیف جلوگیری شود.

**فایده:**

این بررسی به ما کمک می‌کند خریدهای معتبر را از لغو سفارش‌ها و سایر ثبت‌های غیرمعمول جدا کنیم و در مراحل بعدی، ویژگی‌های مشتری و برچسب Churn را بر اساس تعریف قابل دفاعی بسازیم.

</div>


In [100]:

# -----------------------------------------
# 1. ساخت ماسک‌های شناسایی وضعیت تراکنش‌ها
# -----------------------------------------

# فاکتورهایی که شماره آن‌ها با C شروع می‌شود
cancel_mask = (
    df_dedup["InvoiceNo"]
    .str.startswith("C", na=False)
)

# تعداد منفی و صفر
negative_quantity_mask = df_dedup["Quantity"] < 0
zero_quantity_mask = df_dedup["Quantity"] == 0

# قیمت واحد منفی و صفر
negative_price_mask = df_dedup["UnitPrice"] < 0
zero_price_mask = df_dedup["UnitPrice"] == 0


# -----------------------------------------
# 2. ساخت جدول خلاصه وضعیت‌ها
# -----------------------------------------

conditions = {
    "Cancelled Invoices": cancel_mask,
    "Negative Quantity": negative_quantity_mask,
    "Zero Quantity": zero_quantity_mask,
    "Negative UnitPrice": negative_price_mask,
    "Zero UnitPrice": zero_price_mask,
    "Non-positive Quantity": df_dedup["Quantity"] <= 0,
    "Non-positive UnitPrice": df_dedup["UnitPrice"] <= 0,
}

summary = []

for condition_name, mask in conditions.items():
    subset = df_dedup.loc[mask]

    summary.append({
        "Condition": condition_name,
        "Rows": len(subset),
        "Percentage": round(mask.mean() * 100, 2),
        "Unique Invoices": subset["InvoiceNo"].nunique(),
        "Unique Customers": subset["CustomerID"].nunique()
    })

condition_summary = pd.DataFrame(summary)

print("Transaction Quality Summary:")
display(condition_summary)


# -----------------------------------------
# 3. بررسی هم‌پوشانی شرایط
# -----------------------------------------

overlap_summary = pd.DataFrame({
    "Overlap": [
        "Cancelled Invoice & Negative Quantity",
        "Cancelled Invoice & Negative UnitPrice",
        "Negative Quantity & Negative UnitPrice",
        "Cancelled Invoice & Non-positive Quantity",
        "Cancelled Invoice & Non-positive UnitPrice",
    ],
    "Rows": [
        (cancel_mask & negative_quantity_mask).sum(),
        (cancel_mask & negative_price_mask).sum(),
        (negative_quantity_mask & negative_price_mask).sum(),
        (cancel_mask & (df_dedup["Quantity"] <= 0)).sum(),
        (cancel_mask & (df_dedup["UnitPrice"] <= 0)).sum(),
    ]
})

print("\nOverlap Analysis:")
display(overlap_summary)


# -----------------------------------------
# 4. مشاهده نمونه ردیف‌های هر گروه
# -----------------------------------------

sample_columns = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country"
]

print("\nCancelled Invoice Samples:")
display(
    df_dedup.loc[cancel_mask, sample_columns].head(10)
)

print("\nNegative Quantity Samples:")
display(
    df_dedup.loc[negative_quantity_mask, sample_columns].head(10)
)

print("\nNegative UnitPrice Samples:")
display(
    df_dedup.loc[negative_price_mask, sample_columns].head(10)
)

print("\nZero Quantity Samples:")
display(
    df_dedup.loc[zero_quantity_mask, sample_columns].head(10)
)

print("\nZero UnitPrice Samples:")
display(
    df_dedup.loc[zero_price_mask, sample_columns].head(10)
)


Transaction Quality Summary:


,Condition,Rows,Percentage,Unique Invoices,Unique Customers
0,Cancelled Invoices,9251,1.72,3836,1589
1,Negative Quantity,10587,1.97,5172,1589
2,Zero Quantity,0,0.00,0,0
3,Negative UnitPrice,2,0.00,2,0
4,Zero UnitPrice,2510,0.47,2155,31
5,Non-positive Quantity,10587,1.97,5172,1589
6,Non-positive UnitPrice,2512,0.47,2157,31



Overlap Analysis:


,Overlap,Rows
0,Cancelled Invoice & Negative Quantity,9251
1,Cancelled Invoice & Negative UnitPrice,0
2,Negative Quantity & Negative UnitPrice,0
3,Cancelled Invoice & Non-positive Quantity,9251
4,Cancelled Invoice & Non-positive UnitPrice,0



Cancelled Invoice Samples:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,12/1/10 9:41,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,12/1/10 9:49,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,12/1/10 10:24,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,12/1/10 10:24,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,12/1/10 10:24,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,12/1/10 10:24,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,12/1/10 10:24,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,12/1/10 10:24,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,12/1/10 10:24,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,12/1/10 12:38,4.25,17897.0,United Kingdom


... (4 outputs left out)


Zero Quantity Samples:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country



Zero UnitPrice Samples:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,12/1/10 11:52,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,12/1/10 14:32,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,12/1/10 14:33,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,12/1/10 14:33,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,12/1/10 14:34,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,12/1/10 14:34,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,12/1/10 14:34,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,12/1/10 14:35,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,12/1/10 14:35,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,12/1/10 16:50,0.0,NaN,United Kingdom



<div dir="rtl" align="right">

### نتایج اجرا و تحلیل سؤال چهارم: بررسی فاکتورهای لغوشده و مقادیر منفی یا صفر

پس از اجرای بررسی روی دیتاست `df_dedup`، وضعیت فاکتورهای لغوشده، تعداد کالا و قیمت واحد مشخص شد.

**۲. تحلیل فاکتورهای لغوشده**

تعداد 9,251 ردیف متعلق به فاکتورهایی است که شماره آن‌ها با حرف `C` شروع می‌شود. این ردیف‌ها به 3,836 شماره فاکتور یکتا تعلق دارند.

بنابراین باید میان تعداد خطوط تراکنش لغوشده و تعداد فاکتورهای لغوشده تفاوت قائل شد.

**۳. تحلیل Quantity منفی**

تعداد 10,587 ردیف دارای مقدار منفی در ستون `Quantity` بودند.

بررسی هم‌پوشانی نشان داد که تمام 9,251 ردیف فاکتورهای لغوشده در این گروه قرار دارند.

در نتیجه، تعداد 1,336 ردیف دارای Quantity منفی خارج از فاکتورهای لغوشده قرار می‌گیرند.

این ردیف‌ها می‌توانند نشان‌دهنده مرجوعی کالا یا سایر اصلاحات تراکنشی باشند؛ بنابراین صرفاً بر اساس منفی بودن Quantity نمی‌توان ماهیت دقیق آن‌ها را تعیین کرد.

**۴. تحلیل UnitPrice منفی**

تنها دو ردیف دارای قیمت واحد منفی شناسایی شدند. در هر دو نمونه، مقدار Description برابر `Adjust bad debt` و مقدار StockCode برابر `B` بود.

همچنین CustomerID هر دو ردیف گمشده بود.

این الگو نشان می‌دهد که این دو ردیف به احتمال زیاد ثبت‌های تعدیلی حسابداری هستند و نباید به‌عنوان خرید یا مرجوعی معمول کالا در محاسبات مشتری‌محور استفاده شوند.

**۵. تحلیل UnitPrice صفر**

تعداد 2,510 ردیف دارای قیمت واحد صفر بودند که در 2,155 فاکتور مشاهده شدند.

از آنجا که این ردیف‌ها ممکن است مربوط به کالاهای رایگان، هدیه یا سایر ثبت‌های خاص باشند، نمی‌توان تمام آن‌ها را بدون بررسی بیشتر نامعتبر تلقی کرد.

**۶. تحلیل هم‌پوشانی**

بررسی هم‌پوشانی نشان داد:

- تعداد 9,251 ردیف به‌طور هم‌زمان دارای فاکتور لغوشده و Quantity منفی بودند.
- هیچ ردیفی به‌طور هم‌زمان دارای فاکتور لغوشده و UnitPrice منفی نبود.
- هیچ ردیفی به‌طور هم‌زمان دارای Quantity منفی و UnitPrice منفی نبود.

**تصمیم نهایی:**

1. فاکتورهای لغوشده از مجموعه خریدهای معتبر مشتریان کنار گذاشته خواهند شد.
2. ردیف‌های دارای Quantity منفی که فاکتور لغوشده ندارند، جداگانه برای شناسایی مرجوعی‌ها بررسی می‌شوند.
3. دو ردیف دارای UnitPrice منفی به‌عنوان ثبت‌های تعدیلی نگهداری می‌شوند، اما وارد محاسبات خرید مشتری نخواهند شد.
4. ردیف‌های دارای UnitPrice صفر پیش از هرگونه حذف یا کنارگذاری بیشتر بررسی می‌شوند.
5. هیچ‌یک از این دسته‌ها در این مرحله از دیتاست خام حذف نمی‌شوند.

**فایده برای پروژه:**

این تفکیک از وارد شدن فاکتورهای لغوشده، تعدیلات مالی و تراکنش‌های غیرمعمول به محاسبات خرید جلوگیری می‌کند و امکان بررسی جداگانه رفتار مرجوعی مشتریان را فراهم می‌سازد.

</div>


In [101]:

# شناسایی ردیف‌های دارای Quantity منفی
# که فاکتورشان با C شروع نمی‌شود

non_cancelled_negative_qty = df_dedup.loc[
    (df_dedup["Quantity"] < 0)
    & (~df_dedup["InvoiceNo"].str.startswith("C", na=False))
].copy()

print("Non-cancelled Negative Quantity Analysis:")

print("Rows:", len(non_cancelled_negative_qty))
print(
    "Unique Invoices:",
    non_cancelled_negative_qty["InvoiceNo"].nunique()
)
print(
    "Unique Customers:",
    non_cancelled_negative_qty["CustomerID"].nunique()
)
print(
    "Missing CustomerID:",
    non_cancelled_negative_qty["CustomerID"].isna().sum()
)

# مشاهده نمونه‌ها
display(
    non_cancelled_negative_qty[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "UnitPrice",
            "CustomerID",
            "Country"
        ]
    ].head(20)
)


# بررسی وضعیت قیمت در همین ردیف‌ها
print("\nUnitPrice Distribution:")

display(
    non_cancelled_negative_qty["UnitPrice"]
    .describe()
    .to_frame()
)

# بررسی تعداد خطوط بر اساس کد کالا
print("\nMost Frequent StockCodes:")

display(
    non_cancelled_negative_qty["StockCode"]
    .value_counts()
    .head(15)
    .to_frame("Rows")
)


Non-cancelled Negative Quantity Analysis:
Rows: 1336
Unique Invoices: 1336
Unique Customers: 0
Missing CustomerID: 1336


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,12/1/10 16:50,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,12/2/10 14:42,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,12/3/10 15:30,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,12/3/10 15:30,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,12/3/10 15:30,0.0,NaN,United Kingdom
7192,537000,21414,NaN,-22,12/3/10 15:32,0.0,NaN,United Kingdom
7193,537001,21653,NaN,-6,12/3/10 15:33,0.0,NaN,United Kingdom
7195,537003,85126,NaN,-2,12/3/10 15:33,0.0,NaN,United Kingdom
7196,537004,21814,NaN,-30,12/3/10 15:34,0.0,NaN,United Kingdom
7197,537005,21692,NaN,-70,12/3/10 15:35,0.0,NaN,United Kingdom



UnitPrice Distribution:


,UnitPrice
count,1336.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0



Most Frequent StockCodes:


,Rows
StockCode,
85175,5
21830,5
85172,4
22719,4
82494L,4
72802C,4
21161,3
22162,3
22034,3



<div dir="rtl" align="right">

### نتایج اجرا و تحلیل سؤال چهارم: بررسی فاکتورهای لغوشده و تراکنش‌های دارای مقادیر منفی یا صفر

در این مرحله وضعیت فاکتورهای لغوشده و مقادیر غیرمعمول در ستون‌های `Quantity` و `UnitPrice` بررسی شد.

داده‌های مورد بررسی مربوط به دیتاست پس از حذف رکوردهای کاملاً تکراری، شامل 536,641 ردیف بود.

#### ۱. نتایج بررسی وضعیت تراکنش‌ها

| وضعیت | تعداد ردیف | درصد کل | فاکتور یکتا | مشتری یکتا |
|---|---:|---:|---:|---:|
| Cancelled Invoices | 9,251 | 1.72% | 3,836 | 1,589 |
| Negative Quantity | 10,587 | 1.97% | 5,172 | 1,589 |
| Zero Quantity | 0 | 0% | 0 | 0 |
| Negative UnitPrice | 2 | 0.00% | 2 | 0 |
| Zero UnitPrice | 2,510 | 0.47% | 2,155 | 31 |

#### ۲. بررسی فاکتورهای لغوشده

تعداد 9,251 ردیف دارای شماره فاکتور با پیشوند `C` شناسایی شد. این ردیف‌ها متعلق به 3,836 فاکتور یکتا هستند.

بنابراین تعداد ردیف‌های لغوشده با تعداد فاکتورهای لغوشده یکسان نیست؛ زیرا هر فاکتور می‌تواند چندین سطر کالایی داشته باشد.

#### ۳. بررسی Quantity منفی

تعداد 10,587 ردیف دارای مقدار منفی در ستون `Quantity` بودند.

بررسی هم‌پوشانی نشان داد که تمام 9,251 ردیف فاکتورهای لغوشده دارای Quantity منفی هستند.

در نتیجه 1,336 ردیف دارای Quantity منفی خارج از فاکتورهای لغوشده قرار گرفتند.

#### ۴. بررسی تکمیلی 1,336 ردیف دارای Quantity منفی

این ردیف‌ها به‌صورت مستقل بررسی شدند و نتایج زیر به دست آمد:

| شاخص | مقدار |
|---|---:|
| تعداد ردیف‌ها | 1,336 |
| تعداد فاکتورهای یکتا | 1,336 |
| تعداد مشتریان یکتا | 0 |
| تعداد CustomerID گمشده | 1,336 |
| میانگین UnitPrice | 0 |
| انحراف معیار UnitPrice | 0 |
| حداقل UnitPrice | 0 |
| حداکثر UnitPrice | 0 |

بررسی نشان داد که تمام 1,336 ردیف دارای قیمت واحد صفر و شناسه مشتری گمشده هستند.

همچنین تعداد فاکتورهای یکتا با تعداد ردیف‌ها برابر است؛ بنابراین هر ردیف به یک شماره فاکتور مستقل تعلق دارد.

با توجه به نبود شناسه مشتری و صفر بودن قیمت، امکان انتساب این تراکنش‌ها به سابقه خرید یا رفتار مرجوعی مشتری مشخص وجود ندارد.

از طرف دیگر، صرفاً بر اساس منفی بودن Quantity نمی‌توان ماهیت این ردیف‌ها را مرجوعی قطعی در نظر گرفت.

بنابراین این گروه با عنوان `Unidentified Negative-Quantity Records` شناسایی می‌شود.

#### ۵. بررسی UnitPrice منفی

دو ردیف دارای قیمت واحد منفی شناسایی شدند.

در هر دو ردیف:

- `StockCode` برابر `B` بود.
- `Description` برابر `Adjust bad debt` بود.
- `Quantity` برابر 1 بود.
- `UnitPrice` برابر 11062.06- بود.
- `CustomerID` گمشده بود.

این مشخصات نشان می‌دهد که این ردیف‌ها با ثبت‌های تعدیلی بدهی ارتباط دارند و نباید به‌عنوان خرید یا مرجوعی معمول کالا وارد محاسبات مشتری‌محور شوند.

#### ۶. بررسی UnitPrice صفر

تعداد 2,510 ردیف دارای UnitPrice صفر شناسایی شد که در 2,155 فاکتور قرار داشتند.

از آنجا که این ردیف‌ها ممکن است مربوط به کالاهای رایگان، هدیه یا ثبت‌های خاص باشند، صفر بودن قیمت به‌تنهایی دلیل کافی برای حذف تمام آن‌ها نیست.

این دسته به بررسی جداگانه نیاز دارد؛ به‌ویژه در مواردی که CustomerID موجود است.

نتایج نشان می‌دهد که تمام ردیف‌های فاکتورهای لغوشده دارای Quantity منفی هستند؛ اما تمام ردیف‌های دارای Quantity منفی، فاکتور لغوشده نیستند.

#### ۸. تصمیم نهایی

1. فاکتورهای لغوشده از مجموعه خریدهای معتبر مشتریان کنار گذاشته می‌شوند.
2. ردیف‌های دارای Quantity منفی خارج از فاکتورهای لغوشده به‌صورت یک گروه مستقل نگهداری می‌شوند و تا زمان شناسایی ماهیتشان، مرجوعی قطعی محسوب نمی‌شوند.
3. دو ردیف دارای UnitPrice منفی به‌عنوان ثبت‌های تعدیلی شناسایی می‌شوند و وارد محاسبات خرید مشتری نخواهند شد.
4. ردیف‌های دارای UnitPrice صفر به‌صورت خودکار حذف نمی‌شوند و برای بررسی بیشتر نگهداری می‌شوند.
5. اطلاعات این گروه‌ها در داده خام حفظ می‌شود و حذف یا کنارگذاری آن‌ها فقط در زمان ساخت دیتاست متناسب با هر مسئله انجام خواهد شد.

**فایده برای پروژه:**

این تفکیک از ورود فاکتورهای لغوشده، ثبت‌های تعدیلی و تراکنش‌های فاقد هویت مشتری به محاسبات خرید جلوگیری می‌کند. همچنین امکان نگهداری مستقل اطلاعات مربوط به مرجوعی و ثبت‌های غیرمعمول را فراهم می‌سازد.

</div>


In [102]:

# بررسی کامل 1336 ردیف با Quantity منفی خارج از فاکتور لغوشده

audit = non_cancelled_negative_qty

description_check = pd.DataFrame({
    "Missing_Count": audit.isna().sum(),
    "Missing_Percentage": (audit.isna().mean() * 100).round(2)
})

print("Missing Values in Unidentified Records:")
display(description_check)

print("\nStockCode and Description Samples:")

display(
    audit.groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .size()
    .sort_values(ascending=False)
    .head(20)
    .rename("Rows")
    .to_frame()
)


Missing Values in Unidentified Records:


,Missing_Count,Missing_Percentage
InvoiceNo,0,0.00
StockCode,0,0.00
Description,862,64.52
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,1336,100.00
Country,0,0.00



StockCode and Description Samples:


,,Rows
StockCode,Description,
21161,NaN,3
22162,NaN,3
22034,NaN,3
20892,NaN,3
20966,NaN,3
22114,NaN,3
72802B,check,3
72802C,check,3
84631,NaN,3



<div dir="rtl" align="right">

#### ۹. بررسی تکمیلی مقادیر گمشده در رکوردهای دارای Quantity منفی خارج از فاکتورهای لغوشده

به‌منظور بررسی دقیق‌تر 1,336 ردیف دارای Quantity منفی که شماره فاکتور آن‌ها با حرف C شروع نمی‌شد، وضعیت مقادیر گمشده و ترکیب کد کالا و توضیحات بررسی شد.

**نتایج بررسی Missing Values:**

| ستون | تعداد مقادیر گمشده | درصد گمشدگی |
|---|---:|---:|
| InvoiceNo | 0 | 0% |
| StockCode | 0 | 0% |
| Description | 862 | 64.52% |
| Quantity | 0 | 0% |
| InvoiceDate | 0 | 0% |
| UnitPrice | 0 | 0% |
| CustomerID | 1,336 | 100% |
| Country | 0 | 0% |

نتایج نشان داد که تمام 1,336 ردیف فاقد `CustomerID` هستند. همچنین در 862 ردیف مقدار `Description` گمشده است؛ درحالی‌که 474 ردیف دارای توضیحات کالا هستند.

بنابراین، نبود توضیحات کالا در تمام این گروه یکسان نیست و نمی‌توان تمام ردیف‌ها را از نظر اطلاعات توصیفی در یک وضعیت قرار داد.

**بررسی StockCode و Description:**

در بررسی ترکیب کد کالا و توضیحات، نمونه‌هایی مشاهده شد که مقدار `Description` برابر `check` بود؛ از جمله کدهای `72802B` و `72802C`.

این موارد برای بررسی کدهای خاص و ثبت‌های غیرمعمول در مراحل بعد نگهداری می‌شوند و بدون شواهد بیشتر به‌عنوان کالای معمولی یا غیرکالایی طبقه‌بندی نمی‌شوند.

**تحلیل نهایی:**

با توجه به اینکه تمام این ردیف‌ها فاقد شناسه مشتری هستند، امکان اتصال آن‌ها به سوابق خرید مشتریان وجود ندارد. بنابراین نمی‌توان از آن‌ها برای محاسبه LRFM، ساخت ویژگی‌های مشتری یا تعیین وضعیت Churn استفاده کرد.

از طرف دیگر، وجود `StockCode` و اطلاعات زمانی این امکان را فراهم می‌کند که این رکوردها در بررسی عمومی کیفیت داده و شناسایی کدهای خاص حفظ شوند.

**تصمیم نهایی:**

این 1,336 ردیف در داده خام نگهداری می‌شوند، اما در ساخت دیتاست‌های مشتری‌محور مورد استفاده قرار نمی‌گیرند. همچنین برای آن‌ها شناسه مشتری ساختگی یا توضیحات جایگزین تولید نمی‌شود.

**نتیجه سؤال چهارم:**

فاکتورهای لغوشده، تراکنش‌های دارای تعداد منفی، تعدیلات دارای قیمت منفی، ردیف‌های دارای قیمت صفر و رکوردهای منفی فاقد هویت مشتری از یکدیگر تفکیک شدند. این تفکیک مبنای تعریف خرید معتبر در مراحل بعدی Data Preparation خواهد بود.

</div>


---


<div dir="rtl" align="right">

### سؤال پنجم: کدام کدهای کالا نماینده فروش واقعی محصول نیستند؟

**چرایی و دلیل:**

در دیتاست Online Retail تمام مقادیر ستون `StockCode` الزاماً مربوط به کالاهای فیزیکی فروخته‌شده نیستند. برخی ردیف‌ها ممکن است به خدمات پستی، هزینه‌های جانبی، تخفیف‌ها، ثبت‌های دستی، هزینه‌های بانکی یا تعدیلات حسابداری مربوط باشند.

برای مثال، کدهایی مانند `POST`، `DOT`، `M`، `BANK CHARGES`، `AMAZONFEE`، `CRUK` و `D` نیازمند بررسی جداگانه هستند.

ورود بدون بررسی این موارد به محاسبات مشتری ممکن است باعث شود مقدار خرید، تعداد محصولات یکتا و ارزش مالی مشتری به‌درستی محاسبه نشوند.

**هدف:**

شناسایی کدهای غیرکالایی و ثبت‌های خاص، بررسی توضیحات آن‌ها و تعیین ماهیت احتمالی هر گروه.

**منطق تصمیم‌گیری:**

- کدهای شناخته‌شده و مشکوک به‌صورت مستقل بررسی می‌شوند.
- توضیحات کالا برای شناسایی خدمات، هزینه‌ها و تعدیلات جست‌وجو می‌شوند.
- تعداد ردیف‌ها، فاکتورهای یکتا و مشتریان یکتا برای هر کد محاسبه می‌شود.
- مقادیر Quantity و UnitPrice برای تشخیص الگوهای غیرعادی بررسی می‌شوند.
- کدهایی که ماهیت آن‌ها مشخص نیست، تا زمان بررسی بیشتر حذف نمی‌شوند.
- هزینه‌های جانبی مانند پست از فروش کالا تفکیک می‌شوند تا بتوان درباره نحوه محاسبه Monetary تصمیم گرفت.

**فایده:**

جلوگیری از اشتباه گرفتن فروش کالا با خدمات و ثبت‌های مالی و فراهم‌کردن مبنایی دقیق‌تر برای ساخت ویژگی‌های LRFM، به‌ویژه Monetary و Frequency.

</div>


In [103]:

import pandas as pd

# -----------------------------------------
# 1. آماده‌سازی موقت برای جست‌وجوی کدها
# -----------------------------------------

stock_norm = (
    df_dedup["StockCode"]
    .astype("string")
    .str.strip()
    .str.upper()
)

description_norm = (
    df_dedup["Description"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# -----------------------------------------
# 2. فهرست کدهای خاص مورد بررسی
# -----------------------------------------

special_codes = [
    "POST",
    "DOT",
    "M",
    "BANK CHARGES",
    "AMAZONFEE",
    "CRUK",
    "D",
    "B",
    "C2",
    "PADS"
]


# -----------------------------------------
# 3. بررسی هر کد به‌صورت مستقل
# -----------------------------------------

code_results = []

for code in special_codes:

    mask = stock_norm.eq(code)
    subset = df_dedup.loc[mask]

    code_results.append({
        "StockCode": code,
        "Rows": len(subset),
        "UniqueInvoices": subset["InvoiceNo"].nunique(),
        "UniqueCustomers": subset["CustomerID"].nunique(),
        "MissingCustomerID": subset["CustomerID"].isna().sum(),
        "NegativeQuantityRows": (subset["Quantity"] < 0).sum(),
        "ZeroUnitPriceRows": (subset["UnitPrice"] == 0).sum(),
        "PositiveQuantityPositivePriceRows": (
            (subset["Quantity"] > 0)
            & (subset["UnitPrice"] > 0)
        ).sum()
    })


special_code_summary = pd.DataFrame(code_results)

print("Special StockCode Summary:")
display(special_code_summary)


Special StockCode Summary:


,StockCode,Rows,UniqueInvoices,UniqueCustomers,MissingCustomerID,NegativeQuantityRows,ZeroUnitPriceRows,PositiveQuantityPositivePriceRows
0,POST,1256,1254,379,60,126,4,1126
1,DOT,710,710,1,694,1,3,706
2,M,567,518,276,107,244,6,317
3,BANK CHARGES,37,36,10,25,25,0,12
4,AMAZONFEE,34,34,0,34,32,0,2
5,CRUK,16,16,1,0,16,0,0
6,D,77,65,24,0,77,0,0
7,B,3,3,0,3,0,0,1
8,C2,144,144,29,10,2,1,141
9,PADS,4,4,4,0,0,1,3


In [104]:

# -----------------------------------------
# 1. انتخاب ردیف‌های دارای کدهای خاص
# -----------------------------------------

special_code_mask = stock_norm.isin(special_codes)

special_code_rows = df_dedup.loc[
    special_code_mask,
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country"
    ]
].copy()


print("Special StockCode Samples:")
display(
    special_code_rows
    .sort_values(["StockCode", "InvoiceNo"])
    .head(50)
)


# -----------------------------------------
# 2. بررسی نمونه‌هایی که Description مشکوک دارند
# -----------------------------------------

print("\nSpecial Description Samples:")

display(
    description_matches
    .drop_duplicates(subset=["StockCode", "Description"])
    .head(50)
)

Special StockCode Samples:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
15017,537632,AMAZONFEE,AMAZON FEE,1,12/7/10 15:08,13541.330,NaN,United Kingdom
135534,547901,AMAZONFEE,AMAZON FEE,1,3/28/11 11:57,219.760,NaN,United Kingdom
14514,C537600,AMAZONFEE,AMAZON FEE,-1,12/7/10 12:41,1.000,NaN,United Kingdom
15016,C537630,AMAZONFEE,AMAZON FEE,-1,12/7/10 15:04,13541.330,NaN,United Kingdom
16232,C537644,AMAZONFEE,AMAZON FEE,-1,12/7/10 15:34,13474.790,NaN,United Kingdom
16313,C537647,AMAZONFEE,AMAZON FEE,-1,12/7/10 15:41,5519.250,NaN,United Kingdom
16356,C537651,AMAZONFEE,AMAZON FEE,-1,12/7/10 15:49,13541.330,NaN,United Kingdom
16357,C537652,AMAZONFEE,AMAZON FEE,-1,12/7/10 15:51,6706.710,NaN,United Kingdom
43702,C540117,AMAZONFEE,AMAZON FEE,-1,1/5/11 9:55,16888.020,NaN,United Kingdom
43703,C540118,AMAZONFEE,AMAZON FEE,-1,1/5/11 9:57,16453.710,NaN,United Kingdom



Special Description Samples:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
45,536370,POST,POSTAGE,3,12/1/10 8:45,18.00,12583.0,France
53,536373,37370,RETRO COFFEE MUGS ASSORTED,6,12/1/10 9:02,1.06,17850.0,United Kingdom
141,C536379,D,Discount,-1,12/1/10 9:41,27.50,14527.0,United Kingdom
1206,536530,21662,VINTAGE GLASS COFFEE CADDY,1,12/1/10 13:21,5.95,17905.0,United Kingdom
1423,536540,C2,CARRIAGE,1,12/1/10 14:05,50.00,14911.0,EIRE
1546,536544,22300,COFFEE MUG DOG + BALL DESIGN,1,12/1/10 14:32,5.06,NaN,United Kingdom
1547,536544,22301,COFFEE MUG CAT + BIRD DESIGN,2,12/1/10 14:32,5.06,NaN,United Kingdom
1814,536544,DOT,DOTCOM POSTAGE,1,12/1/10 14:32,569.77,NaN,United Kingdom
1922,536544,21216,"SET 3 RETROSPOT TEA,COFFEE,SUGAR",1,12/1/10 14:32,11.02,NaN,United Kingdom
1943,536544,21407,BROWN CHECK CAT DOORSTOP,2,12/1/10 14:32,8.47,NaN,United Kingdom


In [105]:

# -----------------------------------------
# 1. آماده‌سازی کدها و توضیحات
# -----------------------------------------

special_codes = [
    "POST",
    "DOT",
    "M",
    "BANK CHARGES",
    "AMAZONFEE",
    "CRUK",
    "D",
    "B",
    "C2",
    "PADS"
]

# نرمال‌سازی موقت StockCode
stock_norm = (
    df_dedup["StockCode"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# نرمال‌سازی موقت Description
description_norm = (
    df_dedup["Description"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# -----------------------------------------
# 2. انتخاب تمام ردیف‌های کدهای خاص
# -----------------------------------------

special_rows = df_dedup.loc[
    stock_norm.isin(special_codes)
].copy()

special_rows["Description_Normalized"] = (
    special_rows["Description"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# -----------------------------------------
# 3. بررسی تمام ترکیب‌های StockCode و Description
# -----------------------------------------

special_description_details = (
    special_rows
    .groupby(
        ["StockCode", "Description_Normalized"],
        dropna=False
    )
    .agg(
        Rows=("InvoiceNo", "size"),
        UniqueInvoices=("InvoiceNo", "nunique"),
        UniqueCustomers=("CustomerID", "nunique"),
        MissingCustomerID=("CustomerID", lambda x: x.isna().sum()),
        QuantitySum=("Quantity", "sum"),
        NegativeQuantityRows=("Quantity", lambda x: (x < 0).sum()),
        ZeroUnitPriceRows=("UnitPrice", lambda x: (x == 0).sum()),
        MeanUnitPrice=("UnitPrice", "mean"),
        MinUnitPrice=("UnitPrice", "min"),
        MaxUnitPrice=("UnitPrice", "max")
    )
    .reset_index()
    .sort_values(
        ["StockCode", "Rows"],
        ascending=[True, False]
    )
)

print("Special StockCode & Description Analysis:")
display(special_description_details)


# -----------------------------------------
# 4. ساخت مجدد description_matches
# -----------------------------------------

description_pattern = (
    r"POSTAGE|DOTCOM|DISCOUNT|BANK CHARGES|"
    r"AMAZONFEE|CRUK|MANUAL|BAD DEBT|ADJUST|"
    r"CHECK|CARRIAGE|FEE|CHARGES?"
)

description_mask = description_norm.str.contains(
    description_pattern,
    regex=True,
    na=False
)

description_matches = df_dedup.loc[
    description_mask,
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country"
    ]
].copy()

print("\nRows with Special Description Keywords:")
print(len(description_matches))

print("\nDescription Keyword Samples:")
display(description_matches.head(50))


Special StockCode & Description Analysis:


,StockCode,Description_Normalized,Rows,UniqueInvoices,UniqueCustomers,MissingCustomerID,QuantitySum,NegativeQuantityRows,ZeroUnitPriceRows,MeanUnitPrice,MinUnitPrice,MaxUnitPrice
0,AMAZONFEE,AMAZON FEE,34,34,0,34,-30,32,0,7324.784706,1.000,17836.460
1,B,ADJUST BAD DEBT,3,3,0,3,3,0,0,-3687.353333,-11062.060,11062.060
2,BANK CHARGES,BANK CHARGES,37,36,10,25,-13,25,0,202.855162,0.001,1050.150
3,C2,CARRIAGE,143,143,29,9,140,2,0,49.636364,15.000,150.000
4,C2,<NA>,1,1,0,1,150,0,1,0.000000,0.000,0.000
5,CRUK,CRUK COMMISSION,16,16,1,0,-16,16,0,495.839375,1.600,1100.440
6,D,DISCOUNT,77,65,24,0,-1194,77,0,72.484545,0.010,1867.860
7,DOT,DOTCOM POSTAGE,709,709,1,693,707,1,2,290.905585,0.000,4505.170
8,DOT,<NA>,1,1,0,1,1000,0,1,0.000000,0.000,0.000
9,M,MANUAL,566,517,276,106,2924,244,6,378.870866,0.000,38970.000



Rows with Special Description Keywords:
7151

Description Keyword Samples:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
45,536370,POST,POSTAGE,3,12/1/10 8:45,18.00,12583.0,France
53,536373,37370,RETRO COFFEE MUGS ASSORTED,6,12/1/10 9:02,1.06,17850.0,United Kingdom
70,536375,37370,RETRO COFFEE MUGS ASSORTED,6,12/1/10 9:32,1.06,17850.0,United Kingdom
141,C536379,D,Discount,-1,12/1/10 9:41,27.50,14527.0,United Kingdom
283,536396,37370,RETRO COFFEE MUGS ASSORTED,6,12/1/10 10:51,1.06,17850.0,United Kingdom
386,536403,POST,POSTAGE,1,12/1/10 11:27,15.00,12791.0,Netherlands
420,536406,37370,RETRO COFFEE MUGS ASSORTED,6,12/1/10 11:33,1.06,17850.0,United Kingdom
1044,536522,37370,RETRO COFFEE MUGS ASSORTED,2,12/1/10 12:49,1.25,15012.0,United Kingdom
1123,536527,POST,POSTAGE,1,12/1/10 13:04,18.00,12662.0,Germany
1206,536530,21662,VINTAGE GLASS COFFEE CADDY,1,12/1/10 13:21,5.95,17905.0,United Kingdom


In [106]:

# =========================================
# 1. ساخت نسخه نرمال‌شده کد کالا و توضیحات
# =========================================

special_rows = df_dedup.copy()

special_rows["StockCode_Normalized"] = (
    special_rows["StockCode"]
    .astype("string")
    .str.strip()
    .str.upper()
)

special_rows["Description_Normalized"] = (
    special_rows["Description"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# =========================================
# 2. یکسان‌سازی M و m
# =========================================

special_codes = [
    "POST", "DOT", "M", "BANK CHARGES",
    "AMAZONFEE", "CRUK", "D", "B", "C2", "PADS"
]

special_rows_filtered = special_rows.loc[
    special_rows["StockCode_Normalized"].isin(special_codes)
].copy()


# =========================================
# 3. گروه‌بندی با StockCode نرمال‌شده
# =========================================

normalized_description_summary = (
    special_rows_filtered
    .groupby(
        ["StockCode_Normalized", "Description_Normalized"],
        dropna=False
    )
    .agg(
        Rows=("InvoiceNo", "size"),
        UniqueInvoices=("InvoiceNo", "nunique"),
        UniqueCustomers=("CustomerID", "nunique"),
        MissingCustomerID=("CustomerID", lambda x: x.isna().sum()),
        QuantitySum=("Quantity", "sum"),
        NegativeQuantityRows=("Quantity", lambda x: (x < 0).sum()),
        ZeroUnitPriceRows=("UnitPrice", lambda x: (x == 0).sum()),
        MeanUnitPrice=("UnitPrice", "mean"),
        MinUnitPrice=("UnitPrice", "min"),
        MaxUnitPrice=("UnitPrice", "max")
    )
    .reset_index()
    .sort_values(
        ["StockCode_Normalized", "Rows"],
        ascending=[True, False]
    )
)

display(normalized_description_summary)


# =========================================
# 4. اصلاح Regex با مرز کلمات
# =========================================

description_pattern = (
    r"\b(?:"
    r"POSTAGE|"
    r"DOTCOM|"
    r"DISCOUNT|"
    r"BANK\s+CHARGES|"
    r"AMAZON\s+FEE|"
    r"AMAZONFEE|"
    r"CRUK\s+COMMISSION|"
    r"MANUAL|"
    r"BAD\s+DEBT|"
    r"ADJUST(?:MENT)?|"
    r"CHECK|"
    r"CARRIAGE|"
    r"FEE|"
    r"CHARGES?"
    r")\b"
)

description_norm = (
    df_dedup["Description"]
    .astype("string")
    .str.strip()
    .str.upper()
)

description_mask_corrected = description_norm.str.contains(
    description_pattern,
    regex=True,
    na=False
)

description_matches_corrected = df_dedup.loc[
    description_mask_corrected
].copy()

print(
    "Rows with corrected special descriptions:",
    len(description_matches_corrected)
)

print("\nCorrected Description Samples:")

display(
    description_matches_corrected[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "CustomerID"
        ]
    ].head(30)
)


# =========================================
# 5. بررسی ردیف‌هایی که Description خاص دارند
# =========================================

display(
    description_matches_corrected
    .groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .size()
    .sort_values(ascending=False)
    .head(30)
    .rename("Rows")
    .to_frame()
)


,StockCode_Normalized,Description_Normalized,Rows,UniqueInvoices,UniqueCustomers,MissingCustomerID,QuantitySum,NegativeQuantityRows,ZeroUnitPriceRows,MeanUnitPrice,MinUnitPrice,MaxUnitPrice
0,AMAZONFEE,AMAZON FEE,34,34,0,34,-30,32,0,7324.784706,1.000,17836.460
1,B,ADJUST BAD DEBT,3,3,0,3,3,0,0,-3687.353333,-11062.060,11062.060
2,BANK CHARGES,BANK CHARGES,37,36,10,25,-13,25,0,202.855162,0.001,1050.150
3,C2,CARRIAGE,143,143,29,9,140,2,0,49.636364,15.000,150.000
4,C2,<NA>,1,1,0,1,150,0,1,0.000000,0.000,0.000
5,CRUK,CRUK COMMISSION,16,16,1,0,-16,16,0,495.839375,1.600,1100.440
6,D,DISCOUNT,77,65,24,0,-1194,77,0,72.484545,0.010,1867.860
7,DOT,DOTCOM POSTAGE,709,709,1,693,707,1,2,290.905585,0.000,4505.170
8,DOT,<NA>,1,1,0,1,1000,0,1,0.000000,0.000,0.000
9,M,MANUAL,567,518,276,107,2925,244,6,378.207160,0.000,38970.000


Rows with corrected special descriptions: 3522

Corrected Description Samples:


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID
45,536370,POST,POSTAGE,3,18.00,12583.0
141,C536379,D,Discount,-1,27.50,14527.0
386,536403,POST,POSTAGE,1,15.00,12791.0
1123,536527,POST,POSTAGE,1,18.00,12662.0
1423,536540,C2,CARRIAGE,1,50.00,14911.0
1814,536544,DOT,DOTCOM POSTAGE,1,569.77,NaN
1943,536544,21407,BROWN CHECK CAT DOORSTOP,2,8.47,NaN
2239,536569,M,Manual,1,1.25,16274.0
2250,536569,M,Manual,1,18.95,16274.0
3041,536592,DOT,DOTCOM POSTAGE,1,607.49,NaN


,,Rows
StockCode,Description,
POST,POSTAGE,1252
DOT,DOTCOM POSTAGE,709
M,Manual,566
C2,CARRIAGE,143
21407,BROWN CHECK CAT DOORSTOP,139
23444,Next Day Carriage,80
D,Discount,77
23099,FRENCH CARRIAGE LANTERN,63
84795B,SUNSET CHECK HAMMOCK,48



<div dir="rtl" align="right">

### نتایج اجرا و تحلیل سؤال پنجم: شناسایی کدهای غیرکالایی، خدمات و تعدیلات مالی

در این مرحله کدهای خاص ستون `StockCode` با استفاده از اطلاعات توضیحات کالا، تعداد، قیمت واحد و شناسه مشتری بررسی شدند.

برای جلوگیری از تفکیک اشتباه کدهایی که فقط در بزرگی و کوچکی حروف تفاوت داشتند، کدهای کالا به‌صورت موقت نرمال‌سازی شدند. در نتیجه، کدهای `M` و `m` در یک گروه قرار گرفتند.

**۱. نتایج بررسی کدها**

| StockCode | Description | Rows | UniqueInvoices | UniqueCustomers | MissingCustomerID |
|---|---|---:|---:|---:|---:|
| AMAZONFEE | AMAZON FEE | 34 | 34 | 0 | 34 |
| B | ADJUST BAD DEBT | 3 | 3 | 0 | 3 |
| BANK CHARGES | BANK CHARGES | 37 | 36 | 10 | 25 |
| C2 | CARRIAGE | 143 | 143 | 29 | 9 |
| C2 | Missing | 1 | 1 | 0 | 1 |
| CRUK | CRUK COMMISSION | 16 | 16 | 1 | 0 |
| D | DISCOUNT | 77 | 65 | 24 | 0 |
| DOT | DOTCOM POSTAGE | 709 | 709 | 1 | 693 |
| DOT | Missing | 1 | 1 | 0 | 1 |
| M | MANUAL | 567 | 518 | 276 | 107 |
| PADS | PADS TO MATCH ALL CUSHIONS | 4 | 4 | 4 | 0 |
| POST | POSTAGE | 1,252 | 1,250 | 379 | 56 |
| POST | Missing | 4 | 4 | 0 | 4 |

**۲. شناسایی خدمات حمل‌ونقل**

کدهای `POST`، `DOT` و `C2` به‌ترتیب دارای توضیحات `POSTAGE`، `DOTCOM POSTAGE` و `CARRIAGE` هستند.

در مجموع 2,104 ردیف با توضیحات مشخص مرتبط با هزینه ارسال و حمل‌ونقل شناسایی شدند.

این ردیف‌ها نماینده محصولات فروخته‌شده نیستند؛ بنابراین از محاسبه تعداد محصولات، Frequency خرید کالا و Product Monetary جدا می‌شوند.

در صورت موجود بودن CustomerID، اطلاعات هزینه ارسال می‌تواند به‌عنوان ویژگی مستقل نگهداری شود.

**۳. شناسایی تخفیف**

77 ردیف با کد `D` و توضیح `DISCOUNT` شناسایی شدند.

تمام این ردیف‌ها دارای Quantity منفی هستند؛ اما این مقدار منفی نشان‌دهنده ثبت تخفیف است و نباید با مرجوعی کالا اشتباه گرفته شود.

بنابراین این ردیف‌ها از محاسبه تعداد کالا جدا شده و اطلاعات تخفیف به‌صورت مستقل نگهداری می‌شود.

**۴. شناسایی هزینه‌ها و تعدیلات مالی**

کدهای `AMAZONFEE`، `BANK CHARGES`، `B` و `CRUK` به‌ترتیب نماینده کارمزد آمازون، هزینه بانکی، تعدیل بدهی و کمیسیون خیریه هستند.

مجموع این گروه‌ها 90 ردیف است.

این ردیف‌ها وارد محاسبات خرید محصول و Product Monetary نمی‌شوند، اما اطلاعات آن‌ها در داده خام حفظ می‌شود.

**۵. بررسی ثبت‌های Manual**

کد `M` پس از نرمال‌سازی شامل 567 ردیف و 518 فاکتور یکتا است.

این گروه دارای 244 ردیف با Quantity منفی و 107 ردیف فاقد CustomerID است. همچنین دامنه UnitPrice از صفر تا 38,970 متغیر است.

به دلیل وجود الگوهای متفاوت، ماهیت تمام این ثبت‌ها را نمی‌توان صرفاً بر اساس توضیح `MANUAL` تعیین کرد.

بنابراین این گروه تا زمان بررسی دقیق‌تر از محاسبات خرید کالایی جدا نگه داشته می‌شود.

**۶. بررسی کد PADS و توضیحات گمشده**

چهار ردیف با کد `PADS` و توضیح `PADS TO MATCH ALL CUSHIONS` مشاهده شد.

همچنین شش ردیف از کدهای `C2`، `DOT` و `POST` فاقد Description بودند.

این موارد تا زمان مشخص‌شدن ماهیت، در گروه ثبت‌های خاص یا نامشخص نگهداری می‌شوند.

**۷. محدودیت جست‌وجوی مبتنی بر Regex**

جست‌وجوی کلمات مشکوک با Regex تعداد 3,522 ردیف را شناسایی کرد؛ اما بررسی نمونه‌ها نشان داد برخی نتایج، محصولات معمولی هستند که کلماتی مانند `CHECK` یا `CARRIAGE` در نام آن‌ها وجود دارد.

بنابراین تعداد 3,522 به‌عنوان تعداد تراکنش‌های غیرکالایی در نظر گرفته نمی‌شود.

در این پروژه، طبقه‌بندی نهایی بر اساس ترکیب کد کالا و توضیحات معتبر انجام می‌شود و جست‌وجوی متنی صرفاً برای شناسایی موارد نیازمند بررسی بیشتر کاربرد دارد.

**تصمیم نهایی فعلی:**

1. هزینه‌های ارسال و حمل‌ونقل از فروش کالا تفکیک می‌شوند.
2. تخفیف در ویژگی مستقل Discount نگهداری می‌شود.
3. هزینه‌های مالی و تعدیلات حسابداری از محاسبات خرید کالا کنار گذاشته می‌شوند.
4. ثبت‌های Manual و PADS تا زمان بررسی بیشتر در گروه موارد نامشخص باقی می‌مانند.
5. ردیف‌های دارای توضیحات گمشده به‌صورت خودکار طبقه‌بندی نمی‌شوند.
6. هیچ‌یک از ردیف‌ها از دیتاست خام حذف نمی‌شوند.

**نتیجه:**

مشخص شد که ستون `StockCode` علاوه بر کالاهای واقعی، شامل خدمات ارسال، تخفیف، هزینه‌های مالی و ثبت‌های تعدیلی نیز هست. این تفکیک برای تعریف صحیح خرید معتبر و محاسبه ویژگی‌های LRFM ضروری است.

</div>


In [107]:

# =========================================
# 1. ساخت نسخه نرمال‌شده برای بررسی
# =========================================

df_review = df_dedup.copy()

df_review["StockCode_Normalized"] = (
    df_review["StockCode"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_review["Description_Normalized"] = (
    df_review["Description"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# =========================================
# 2. بررسی موارد خاص با توضیحات دقیق
# =========================================

target_descriptions = [
    "NEXT DAY CARRIAGE",
    "PACKING CHARGE",
    "CHECK",
    "TAIG ADJUST",
    "ADJUSTMENT",
    "PADS TO MATCH ALL CUSHIONS"
]

target_mask = df_review["Description_Normalized"].isin(
    target_descriptions
)

target_details = (
    df_review.loc[target_mask]
    .groupby(
        ["StockCode_Normalized", "Description_Normalized"],
        dropna=False
    )
    .agg(
        Rows=("InvoiceNo", "size"),
        UniqueInvoices=("InvoiceNo", "nunique"),
        UniqueCustomers=("CustomerID", "nunique"),
        MissingCustomerID=("CustomerID", lambda x: x.isna().sum()),
        NegativeQuantityRows=("Quantity", lambda x: (x < 0).sum()),
        QuantitySum=("Quantity", "sum"),
        MeanUnitPrice=("UnitPrice", "mean"),
        MinUnitPrice=("UnitPrice", "min"),
        MaxUnitPrice=("UnitPrice", "max")
    )
    .reset_index()
    .sort_values("Rows", ascending=False)
)

print("Unusual Description Analysis:")
display(target_details)


# =========================================
# 3. بررسی جزئیات ثبت‌های Manual
# =========================================

manual_rows = df_review.loc[
    df_review["StockCode_Normalized"] == "M"
].copy()

print("\nManual Transaction Samples:")

display(
    manual_rows[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "UnitPrice",
            "CustomerID",
            "Country"
        ]
    ].sort_values(
        ["InvoiceNo", "UnitPrice"]
    ).head(30)
)


# =========================================
# 4. بررسی تمام ردیف‌های PADS
# =========================================

pads_rows = df_review.loc[
    df_review["StockCode_Normalized"] == "PADS"
].copy()

print("\nPADS Transactions:")

display(
    pads_rows[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "UnitPrice",
            "CustomerID",
            "Country"
        ]
    ]
)


Unusual Description Analysis:


,StockCode_Normalized,Description_Normalized,Rows,UniqueInvoices,UniqueCustomers,MissingCustomerID,NegativeQuantityRows,QuantitySum,MeanUnitPrice,MinUnitPrice,MaxUnitPrice
108,23444,NEXT DAY CARRIAGE,80,80,63,0,1,78,15.18750,15.0,30.000
109,23574,PACKING CHARGE,16,16,9,0,2,12,7.50000,7.5,7.500
169,PADS,PADS TO MATCH ALL CUSHIONS,4,4,4,0,0,4,0.00075,0.0,0.001
131,72802B,CHECK,3,3,0,3,3,-290,0.00000,0.0,0.000
132,72802C,CHECK,3,3,0,3,3,-75,0.00000,0.0,0.000
...,...,...,...,...,...,...,...,...,...,...,...
163,85208,CHECK,1,1,0,1,1,-22,0.00000,0.0,0.000
165,90012B,CHECK,1,1,0,1,1,-5,0.00000,0.0,0.000
166,90014A,CHECK,1,1,0,1,1,-12,0.00000,0.0,0.000
167,90195A,CHECK,1,1,0,1,1,-45,0.00000,0.0,0.000



Manual Transaction Samples:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2239,536569,M,Manual,1,12/1/10 15:35,1.25,16274.0,United Kingdom
2250,536569,M,Manual,1,12/1/10 15:35,18.95,16274.0,United Kingdom
5684,536865,M,Manual,1,12/3/10 11:28,2.55,NaN,United Kingdom
6798,536981,M,Manual,2,12/3/10 14:26,0.85,14723.0,United Kingdom
7976,537077,M,Manual,12,12/5/10 11:59,0.42,17062.0,United Kingdom
8530,537137,M,Manual,36,12/5/10 12:43,0.85,16327.0,United Kingdom
8587,537140,M,Manual,1,12/5/10 12:53,0.42,12748.0,United Kingdom
8588,537140,M,Manual,1,12/5/10 12:53,0.85,12748.0,United Kingdom
9326,537199,M,Manual,1,12/5/10 14:06,0.65,15894.0,United Kingdom
9575,537208,M,Manual,4,12/5/10 15:12,0.85,15889.0,United Kingdom



PADS Transactions:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
157195,550193,PADS,PADS TO MATCH ALL CUSHIONS,1,4/15/11 9:27,0.001,13952.0,United Kingdom
279045,561226,PADS,PADS TO MATCH ALL CUSHIONS,1,7/26/11 10:13,0.001,15618.0,United Kingdom
358655,568158,PADS,PADS TO MATCH ALL CUSHIONS,1,9/25/11 12:22,0.000,16133.0,United Kingdom
359871,568200,PADS,PADS TO MATCH ALL CUSHIONS,1,9/25/11 14:58,0.001,16198.0,United Kingdom



<div dir="rtl" align="right">

### تکمیل نتایج سؤال پنجم: شناسایی خدمات، ثبت‌های مالی و کدهای غیرکالایی

پس از بررسی کدهای خاص و توضیحات غیرعادی، موارد بیشتری شناسایی شدند.

**۱. خدمات حمل‌ونقل و ارسال**

علاوه بر کدهای `POST`، `DOT` و `C2`، کد `23444` با توضیح `NEXT DAY CARRIAGE` نیز شناسایی شد.

این کد شامل 80 ردیف، 80 فاکتور یکتا و 63 مشتری یکتا است.

با توجه به ماهیت توضیحات، این ردیف‌ها مربوط به خدمت ارسال سریع هستند و نباید به‌عنوان کالای فروخته‌شده در نظر گرفته شوند.

**۲. هزینه بسته‌بندی**

کد `23574` با توضیح `PACKING CHARGE` شامل 16 ردیف است.

این ردیف‌ها دارای قیمت واحد ثابت 7.5 هستند و به هزینه بسته‌بندی اشاره دارند.

بنابراین از محاسبات فروش کالا جدا می‌شوند و در صورت نیاز به‌عنوان ویژگی مستقل هزینه بسته‌بندی نگهداری خواهند شد.

**۳. بررسی PADS**

چهار ردیف با کد `PADS` و توضیح `PADS TO MATCH ALL CUSHIONS` مشاهده شد.

با وجود مثبت بودن Quantity و وجود CustomerID، قیمت این ردیف‌ها صفر یا بسیار نزدیک به صفر است.

به دلیل نامشخص بودن ماهیت دقیق این ثبت‌ها، فعلاً از محاسبات خرید کالایی جدا نگه داشته می‌شوند.

**۴. بررسی CHECK**

برخی ردیف‌ها دارای Description دقیقاً برابر `CHECK` هستند. این ردیف‌ها در نمونه‌های بررسی‌شده دارای Quantity منفی، UnitPrice صفر و CustomerID گمشده بودند.

این الگو با خرید معمولی کالا مطابقت ندارد.

بااین‌حال، وجود کلمه CHECK در نام محصولاتی مانند `BROWN CHECK CAT DOORSTOP` به‌تنهایی نشانه غیرکالایی بودن آن‌ها نیست.

بنابراین فقط ثبت‌هایی که توضیح آن‌ها دقیقاً `CHECK` است در گروه بررسی موارد خاص قرار می‌گیرند.

**۵. تصمیم درباره MANUAL**

کد `M` شامل 567 ردیف با الگوهای متفاوت در Quantity و UnitPrice است.

از آنجا که توضیح `MANUAL` به‌تنهایی ماهیت اقتصادی این ثبت‌ها را مشخص نمی‌کند، تمام این ردیف‌ها فعلاً از محاسبات خرید معتبر جدا نگه داشته می‌شوند.

**۶. تصمیم نهایی**

خدمات ارسال و حمل‌ونقل، هزینه بسته‌بندی، تخفیف، هزینه‌های مالی، تعدیلات حسابداری، کمیسیون و ثبت‌های دستی یا نامشخص از فروش کالایی تفکیک می‌شوند.

اطلاعات این گروه‌ها در داده خام حفظ می‌شود و حذف فیزیکی انجام نمی‌گیرد.

**نتیجه:**

ستون StockCode صرفاً شامل شناسه محصولات نیست و انواع خدمات و ثبت‌های مالی نیز در آن وجود دارند. بنابراین تعریف خرید معتبر باید علاوه بر مثبت بودن Quantity و UnitPrice، ماهیت کالایی ردیف را نیز در نظر بگیرد.

</div>


---


<div dir="rtl" align="right">

### سؤال ششم: تعریف معیارهای خرید معتبر (Valid Purchase Criteria)

**چرایی و دلیل:**

پس از بررسی کیفیت داده و شناسایی خدمات، هزینه‌ها، تخفیف‌ها و تعدیلات مالی مشخص شد که تمام ردیف‌های دارای مقادیر مثبت، الزاماً نشان‌دهنده خرید واقعی کالا نیستند.

بنابراین لازم است معیار مشخصی برای تشخیص خطوط خرید معتبر تعریف شود تا ویژگی‌های مشتریان بر اساس تراکنش‌های کالایی واقعی محاسبه شوند.

**هدف:**

تعریف یک معیار یکپارچه برای شناسایی خطوط خرید معتبر و تفکیک آن‌ها از مرجوعی‌ها، لغو سفارش‌ها، خدمات جانبی و ثبت‌های مالی.

**منطق تصمیم‌گیری:**

یک خط خرید معتبر باید شرایط زیر را داشته باشد:

1. شماره فاکتور موجود و معتبر باشد.
2. فاکتور از نوع لغوشده نباشد.
3. تاریخ تراکنش معتبر باشد.
4. Quantity بزرگ‌تر از صفر باشد.
5. UnitPrice بزرگ‌تر از صفر باشد.
6. CustomerID موجود باشد؛ زیرا هدف، ساخت داده‌های سطح مشتری است.
7. StockCode و Description به گروه خدمات، هزینه‌ها و تعدیلات شناسایی‌شده تعلق نداشته باشند.

**فایده:**

- جلوگیری از ورود خدمات و ثبت‌های مالی به محاسبات فروش کالا.
- محاسبه صحیح Frequency، Monetary و سایر ویژگی‌های LRFM.
- جلوگیری از اشتباه گرفتن مرجوعی با خرید.
- ایجاد یک مبنای مشترک و قابل اعتماد برای آماده‌سازی داده‌های K-Means و XGBoost.

</div>


In [108]:

# =========================================
# 1. آماده سازی ستون های مورد نیاز
# =========================================

df_valid_check = df_dedup.copy()

df_valid_check["InvoiceDate"] = pd.to_datetime(
    df_valid_check["InvoiceDate"],
    errors="coerce"
)

df_valid_check["StockCode_Normalized"] = (
    df_valid_check["StockCode"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_valid_check["Description_Normalized"] = (
    df_valid_check["Description"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# =========================================
# 2. تعریف کدهای غیرکالایی شناسایی شده
# =========================================

non_merchandise_codes = [
    "POST",
    "DOT",
    "C2",
    "23444",
    "23574",
    "BANK CHARGES",
    "AMAZONFEE",
    "CRUK",
    "B",
    "D",
    "M",
    "PADS"
]


# =========================================
# 3. شناسایی ثبت های خاص بر اساس Description
# =========================================

special_descriptions = [
    "POSTAGE",
    "DOTCOM POSTAGE",
    "CARRIAGE",
    "NEXT DAY CARRIAGE",
    "PACKING CHARGE",
    "DISCOUNT",
    "BANK CHARGES",
    "AMAZON FEE",
    "CRUK COMMISSION",
    "ADJUST BAD DEBT",
    "MANUAL",
    "PADS TO MATCH ALL CUSHIONS",
    "CHECK"
]


# =========================================
# 4. ساخت شرط های خرید معتبر
# =========================================

valid_invoice = (
    df_valid_check["InvoiceNo"].notna()
    & ~df_valid_check["InvoiceNo"]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.startswith("C", na=False)
)

valid_date = df_valid_check["InvoiceDate"].notna()

valid_quantity = df_valid_check["Quantity"] > 0

valid_price = df_valid_check["UnitPrice"] > 0

known_customer = df_valid_check["CustomerID"].notna()

valid_stock_code = (
    ~df_valid_check["StockCode_Normalized"]
    .isin(non_merchandise_codes)
)

valid_description = (
    ~df_valid_check["Description_Normalized"]
    .isin(special_descriptions)
)


# =========================================
# 5. ترکیب تمام شرایط
# =========================================

valid_purchase_mask = (
    valid_invoice
    & valid_date
    & valid_quantity
    & valid_price
    & known_customer
    & valid_stock_code
    & valid_description
)


# =========================================
# 6. گزارش تعداد ردیف های هر شرط
# =========================================

conditions_summary = pd.DataFrame({
    "Condition": [
        "Valid Invoice",
        "Valid Date",
        "Positive Quantity",
        "Positive UnitPrice",
        "Known CustomerID",
        "Valid StockCode",
        "Valid Description",
        "All Conditions"
    ],

    "Rows": [
        valid_invoice.sum(),
        valid_date.sum(),
        valid_quantity.sum(),
        valid_price.sum(),
        known_customer.sum(),
        valid_stock_code.sum(),
        valid_description.sum(),
        valid_purchase_mask.sum()
    ]
})

conditions_summary["Percentage"] = (
    conditions_summary["Rows"] / len(df_valid_check) * 100
)

display(conditions_summary)


# =========================================
# 7. بررسی نمونه خریدهای معتبر
# =========================================

valid_purchase_preview = df_valid_check.loc[
    valid_purchase_mask
]

print("Valid Purchase Shape:", valid_purchase_preview.shape)

display(valid_purchase_preview.head(10))


C:\Users\Mega system32894565\AppData\Local\Temp\ipykernel_17980\2644123324.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_valid_check["InvoiceDate"] = pd.to_datetime(


,Condition,Rows,Percentage
0,Valid Invoice,527390,98.276129
1,Valid Date,536641,100.000000
2,Positive Quantity,526054,98.027173
3,Positive UnitPrice,534129,99.531903
4,Known CustomerID,401604,74.836623
5,Valid StockCode,533695,99.451030
6,Valid Description,533541,99.422333
7,All Conditions,391057,72.871249


Valid Purchase Shape: (391057, 10)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,StockCode_Normalized,Description_Normalized
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,85123A,WHITE HANGING HEART T-LIGHT HOLDER
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,71053,WHITE METAL LANTERN
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,84406B,CREAM CUPID HEARTS COAT HANGER
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,84029G,KNITTED UNION FLAG HOT WATER BOTTLE
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,84029E,RED WOOLLY HOTTIE WHITE HEART.
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,22752,SET 7 BABUSHKA NESTING BOXES
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,21730,GLASS STAR FROSTED T-LIGHT HOLDER
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,22633,HAND WARMER UNION JACK
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,22632,HAND WARMER RED POLKA DOT
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom,84879,ASSORTED COLOUR BIRD ORNAMENT


### نتیجه اولیه
پس از اعمال هم‌زمان شروط فوق، تعداد ۳۹۱٬۰۵۷ ردیف از مجموع ۵۳۶٬۶۴۱ ردیف باقی‌مانده پس از حذف تکرارهای دقیق، به‌عنوان خرید معتبر اولیه شناسایی شد. این مقدار معادل حدود ۷۲٫۸۷ درصد داده‌هاست.

### تفسیر
مجموعه استخراج‌شده شامل تراکنش‌هایی است که در آن‌ها امکان انتساب خرید به مشتری مشخص وجود دارد و مقادیر تعداد و قیمت نیز مثبت هستند. این مجموعه مبنای اولیه ساخت داده‌های مشتری‌محور خواهد بود.

بااین‌حال، این تعریف یک مرحله پالایش اولیه است و لازم است کدهای خاص و رکوردهای غیرمعمول باقی‌مانده نیز بررسی شوند تا از ورود خدمات، اصلاحات مالی و سایر تراکنش‌های غیرکالایی به محاسبات رفتار خرید جلوگیری شود.

**نکته:** رکوردهای حذف‌شده از داده خام پاک نمی‌شوند؛ بلکه برای تحلیل‌های مرتبط با مرجوعی، لغو سفارش، خدمات و اصلاحات مالی در مجموعه جداگانه قابل نگهداری هستند.

In [110]:
# حذف رکوردهای تکراری از دیتاست خام
df_no_duplicates = df.drop_duplicates().copy()

print("Shape:", df_no_duplicates.shape)

Shape: (536641, 8)


In [111]:
# نرمال‌سازی کد کالا و توضیحات

df_no_duplicates["StockCode_Normalized"] = (
    df_no_duplicates["StockCode"].astype("string").str.strip().str.upper()
)

df_no_duplicates["Description_Normalized"] = (
    df_no_duplicates["Description"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [113]:
non_merchandise_descriptions = [
    "POSTAGE",
    "DOTCOM POSTAGE",
    "CARRIAGE",
    "NEXT DAY CARRIAGE",
    "PACKING CHARGE",
    "DISCOUNT",
    "BANK CHARGES",
    "AMAZON FEE",
    "CRUK COMMISSION",
    "ADJUST BAD DEBT",
    "MANUAL",
    "PADS TO MATCH ALL CUSHIONS",
    "CHECK"
]

In [114]:
valid_purchase_mask = (
    (~df_no_duplicates["InvoiceNo"].astype("string").str.startswith("C", na=False))
    & df_no_duplicates["InvoiceDate"].notna()
    & (df_no_duplicates["Quantity"] > 0)
    & (df_no_duplicates["UnitPrice"] > 0)
    & df_no_duplicates["CustomerID"].notna()
    & (~df_no_duplicates["StockCode_Normalized"].isin(non_merchandise_codes))
    & (~df_no_duplicates["Description_Normalized"].isin(non_merchandise_descriptions))
)

valid_purchase = df_no_duplicates.loc[valid_purchase_mask].copy()

print("Valid Purchase Shape:", valid_purchase.shape)

Valid Purchase Shape: (391057, 10)


In [115]:
valid_purchase["StockCode_Normalized"].value_counts().head(30)

StockCode_Normalized
85123A    2023
22423     1713
85099B    1615
84879     1395
47566     1389
20725     1304
22720     1152
23203     1091
20727     1078
22383     1063
22197     1060
21212     1050
23298     1021
23209     1020
22086     1003
22382      993
22469      979
20728      977
22457      972
22384      941
21034      897
22727      886
22386      884
82482      882
22993      881
22960      879
23206      878
22961      870
22138      858
22666      836
Name: count, dtype: Int64

In [116]:
valid_purchase["Description_Normalized"].value_counts().head(30)

Description_Normalized
WHITE HANGING HEART T-LIGHT HOLDER    2016
REGENCY CAKESTAND 3 TIER              1713
JUMBO BAG RED RETROSPOT               1615
ASSORTED COLOUR BIRD ORNAMENT         1395
PARTY BUNTING                         1389
LUNCH BAG RED RETROSPOT               1303
SET OF 3 CAKE TINS PANTRY DESIGN      1152
LUNCH BAG  BLACK SKULL.               1078
PACK OF 72 RETROSPOT CAKE CASES       1050
SPOTTY BUNTING                        1014
PAPER CHAIN KIT 50'S CHRISTMAS        1003
LUNCH BAG SPACEBOY DESIGN              993
HEART OF WICKER SMALL                  979
LUNCH BAG CARS BLUE                    977
NATURAL SLATE HEART CHALKBOARD         972
LUNCH BAG PINK POLKADOT                941
LUNCH BAG SUKI DESIGN                  915
REX CASH+CARRY JUMBO SHOPPER           897
ALARM CLOCK BAKELIKE RED               886
JUMBO BAG PINK POLKADOT                884
WOODEN PICTURE FRAME WHITE FINISH      882
SET OF 4 PANTRY JELLY MOULDS           881
JAM MAKING SET WITH JARS       